In [ ]:
!pip install -U openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 22.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=49a0f52c2472244e0e009b22667e1a05b1dfa6f97f02e92ececa6db7be9fd484
  Stored in directory: /root/.cache/pip/wheels/ca/58/d5/fb4539ad74c3ca81eb40f7eda020ac77d080b33ad57449d485
Successfully built openai-whisper


**charger le modèle et transcrire un fichier MP3 :**

In [ ]:
import os

chemin = "/content/extrait.mp3"

print("Le fichier existe :", os.path.exists(chemin))

Le fichier existe : False


In [ ]:
!pip install -q gdown #sert à installer un outil qui permet de télécharger des fichiers depuis Google Drive vers Google Cola

In [ ]:
# Supprimer l'ancienne copie du fichier ZIP dans Colab, si elle existe
!rm -f PROJET_SID_2026.zip

# Télécharger le fichier depuis Google Drive et l'enregistrer sous le nom PROJET_SID_2026.zip
!gdown --fuzzy "https://drive.google.com/file/d/1ykUsEsbMoMlsji4eewSPVgUgqjT98GJH/view?usp=drive_link" -O PROJET_SID_2026.zip

Downloading...
From (original): https://drive.google.com/uc?id=1ykUsEsbMoMlsji4eewSPVgUgqjT98GJH
From (redirected): https://drive.google.com/uc?id=1ykUsEsbMoMlsji4eewSPVgUgqjT98GJH&confirm=t&uuid=7666689f-fa41-44a5-bb32-f0b640bc6c90
To: /content/PROJET_SID_2026.zip
100% 4.00G/4.00G [00:56<00:00, 70.9MB/s]


In [ ]:
# Créer le dossier /content/data s'il n'existe pas déjà
!mkdir -p /content/data

# Décompresser l'archive ZIP dans le dossier /content/data
!unzip -q /content/PROJET_SID_2026.zip -d /content/data

In [ ]:
!ls /content/data
!ls /content/data/*

__MACOSX  PROJET_SID_2026
/content/data/__MACOSX:
PROJET_SID_2026

/content/data/PROJET_SID_2026:
movie


In [ ]:
import os

ROOT = "/content/data/PROJET_SID_2026/movie"
DOSSIER_AUDIO = os.path.join(ROOT, "audio")

print("Dossier audio trouvé :", os.path.exists(DOSSIER_AUDIO))
print("Contenu :", os.listdir(DOSSIER_AUDIO)[:10])

Dossier audio trouvé : True
Contenu : ['00024', '00140', '00135', '00131', '00081', '00224', '00050', '00260', '00065', '00031']


In [ ]:
import glob

fichiers_audio = glob.glob(
    DOSSIER_AUDIO + "/**/*",
    recursive=True
)

fichiers_audio = [
    f for f in fichiers_audio
    if os.path.isfile(f)
    and f.lower().endswith((".mp3", ".wav", ".m4a", ".mp4"))
]

print("Nombre de fichiers audio trouvés :", len(fichiers_audio))
print(fichiers_audio[:5])

Nombre de fichiers audio trouvés : 3088
['/content/data/PROJET_SID_2026/movie/audio/00024/00024_clip_003.mp3', '/content/data/PROJET_SID_2026/movie/audio/00024/00024_clip_009.mp3', '/content/data/PROJET_SID_2026/movie/audio/00024/00024_clip_004.mp3', '/content/data/PROJET_SID_2026/movie/audio/00024/00024_clip_001.mp3', '/content/data/PROJET_SID_2026/movie/audio/00024/00024_clip_007.mp3']


In [ ]:
import whisper

model = whisper.load_model("small")

chemin_audio = fichiers_audio[0]
resultat = model.transcribe(chemin_audio, fp16=False)

print("Fichier :", chemin_audio)
print("Transcription :", resultat["text"])

Fichier : /content/data/PROJET_SID_2026/movie/audio/00024/00024_clip_003.mp3
Transcription : 


**je verifie dabord que le GPU fonctionne**

In [ ]:
!nvidia-smi

Wed Oct 21 09:15:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

**Transcription complete de les fichiers**

In [ ]:

import os
import pandas as pd
import whisper
import torch
from tqdm.auto import tqdm

# 1. Vérifier que le GPU est disponible
assert torch.cuda.is_available(), "GPU non disponible !"

# 2. Charger Whisper sur le GPU
model = whisper.load_model("small", device="cuda")

# 3. Définir le fichier de sauvegarde
fichier_csv = "/content/transcriptions_audio.csv"

# 4. Charger les résultats existants, s'ils existent
if os.path.exists(fichier_csv):
    df_transcriptions = pd.read_csv(fichier_csv)
    deja_traites = set(df_transcriptions["chemin_audio"])
else:
    df_transcriptions = pd.DataFrame(
        columns=["video_id", "chemin_audio", "transcription"]
    )
    deja_traites = set()

# 5. Préparer la liste des fichiers restant à traiter
fichiers_a_traiter = [
    f for f in fichiers_audio if f not in deja_traites
]

print("Total de fichiers :", len(fichiers_audio))
print("Déjà transcrits :", len(deja_traites))
print("Restant à traiter :", len(fichiers_a_traiter))

# 6. Transcrire et sauvegarder progressivement
erreurs = []

for i, chemin_audio in enumerate(
    tqdm(fichiers_a_traiter, desc="Transcription des audios"),
    start=1
):
    try:
        resultat = model.transcribe(chemin_audio)

        video_id = os.path.splitext(
            os.path.basename(chemin_audio)
        )[0]

        df_transcriptions.loc[len(df_transcriptions)] = [
            video_id,
            chemin_audio,
            resultat["text"].strip()
        ]

    except Exception as e:
        erreurs.append({
            "chemin_audio": chemin_audio,
            "erreur": str(e)
        })

    # Sauvegarder toutes les 25 transcriptions
    if i % 25 == 0:
        df_transcriptions.to_csv(fichier_csv, index=False)

# Sauvegarde finale
df_transcriptions.to_csv(fichier_csv, index=False)

print("\nTranscriptions enregistrées :", len(df_transcriptions))
print("Erreurs rencontrées :", len(erreurs))
print("CSV sauvegardé ici :", fichier_csv)


Total de fichiers : 3088
Déjà transcrits : 0
Restant à traiter : 3088


Transcription des audios:   0%|          | 0/3088 [00:00<?, ?it/s]


Transcriptions enregistrées : 3088
Erreurs rencontrées : 0
CSV sauvegardé ici : /content/transcriptions_audio.csv


In [ ]:
import pandas as pd

df_transcriptions = pd.read_csv("/content/transcriptions_audio.csv")

display(df_transcriptions.head(15))   #pour avoir un aperçu

print("Nombre de lignes :", len(df_transcriptions))
print("Colonnes :", df_transcriptions.columns.tolist())

,video_id,chemin_audio,transcription
0,00024_clip_003,/content/data/PROJET_SID_2026/movie/audio/0002...,NaN
1,00024_clip_009,/content/data/PROJET_SID_2026/movie/audio/0002...,Some video and a chick calls you on the phone ...
2,00024_clip_004,/content/data/PROJET_SID_2026/movie/audio/0002...,"God, I'm sorry. Sir, sir, please stay in your ..."
3,00024_clip_001,/content/data/PROJET_SID_2026/movie/audio/0002...,All I gotta do is make it to the next five min...
4,00024_clip_007,/content/data/PROJET_SID_2026/movie/audio/0002...,cute guy sitting right next to us. It turns ou...
5,00024_clip_010,/content/data/PROJET_SID_2026/movie/audio/0002...,NaN
6,00140_clip_019,/content/data/PROJET_SID_2026/movie/audio/0014...,మాసికాసినినినినిని.
7,00140_clip_004,/content/data/PROJET_SID_2026/movie/audio/0014...,"For heaven and heaven, hallelujah, hallelujah,..."
8,00140_clip_011,/content/data/PROJET_SID_2026/movie/audio/0014...,Make it your first care to love one another.
9,00140_clip_015,/content/data/PROJET_SID_2026/movie/audio/0014...,Leave tomorrow to fret over its own needs.


Nombre de lignes : 3088
Colonnes : ['video_id', 'chemin_audio', 'transcription']


In [ ]:
from google.colab import files

files.download("/content/transcriptions_audio.csv") #pour telechier le fichier transcrit

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>